# March Madness: Can we predict the next tournament?

A personal exploration, evolved from an earlier graduate-course notebook. The goal is **pre-game win probability**, not identifying the seed of an already-known winner. We compare a seed-only benchmark with a regular-season-informed model and evaluate each on tournament seasons absent from training.

The source dataset, assumptions, and known weaknesses are described in the project README and `RESEARCH_NOTES.md`.

## 1. Reproducible setup

From the repository root, run `python -m pip install -e .`. Obtain the public NCAA competition CSVs from Kaggle and place them in `data/`. No Kaggle credentials, local home-directory paths, or saved outputs are required inside this notebook.

In [ ]:
from pathlib import Path
from madness_lab.data import load_kaggle_data, build_matchups, FEATURE_COLUMNS
from madness_lab.model import forward_season_validation, summarize_backtest

data_dir = Path('../data')  # Notebook launched from notebooks/
regular, tournament, seeds = load_kaggle_data(data_dir)
matchups = build_matchups(regular, tournament, seeds)
print('Games:', len(matchups), '| Seasons:', matchups.Season.nunique())

## 2. Inspect the features

For each matchup, Team A is the team with the smaller numeric ID, *regardless of who won*. The target is 1 when Team A won. Predictors contain only seeds and aggregated regular-season statistics available before tournament play. Feature differences are signed A minus B (seed advantage is B seed minus A seed, since smaller seeds are stronger).

In [ ]:
display(matchups[['Season', 'TeamAID', 'TeamBID', 'team_a_won', *FEATURE_COLUMNS]].head(8))

## 3. Compare against a fair benchmark

A random game-level split can let similar data from the same season enter both training and test sets. Here, every tested tournament year is evaluated using *only prior tournament years for training*. Even the feature scaler fits only on the training portion. The seed-only model uses the same years, estimator and regularization, isolating what extra season statistics add.

In [ ]:
backtest = forward_season_validation(matchups, min_train_seasons=5, last_n_seasons=5)
display(backtest)
display(summarize_backtest(backtest))

## 4. Interpretation and limitations

- **Accuracy** answers how often a model chooses the right winner at a 50% threshold.
- **Brier score** and **log loss** evaluate the assigned probabilities. Lower is better; confidently wrong picks are penalized.
- These aggregate features do not account for schedule difficulty, roster changes, or injuries. A useful next experiment is a season-by-season opponent-adjusted rating.
- Results are intentionally generated when the notebook runs, not presented as historical claims without real-data verification.